In [ ]:
'''Промт, отправленный в DeepSeek:
Мне нужно сгенерировать с помощью LLM диалог ментора и менти, иллюстрирующий компетенции ментора. 
Компетенции я подготовил в файле .txt. 
Твоя задача - разработать Python-скрипт для генерации этого диалога. 
В качестве LLM я использую oLLAMA-сервер по адресу https://ollama.lourie.info/ позади HTTP аутентификации (пользователь, пароль, в виде кортежа auth = (user, password)). 
Требования к коду: обеспечить возможность retry, проверки статуса готовности oLLAMA-сервера, как вариант: использовать streaming-режим генерации oLLAMA. 
Также стоит помнить, что задержка генерации первого токена будет большой, это стоит учитывать и использовать повышенный (и настраиваемый timeout=300). 
Код будет запускаться в Jupyter, поэтому рекомендую сразу готовить код в виде блоков (ячеек).'''

Пример работы с self-hosted LLM на движке oLLAMA ()

In [1]:
import requests
import json
import time
import sys
from pathlib import Path
from typing import Optional, Dict, Any, Generator
import logging

# Настройка логирования для отладки
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# Конфигурация Ollama-сервера
OLLAMA_BASE_URL = "https://ollama.lourie.info"
#Read credentials from file
with open('Ollama_Creds.json', 'r') as f:
    creds = json.load(f)
    f.close()
AUTH = (creds['User'], creds['Password'])
MODEL_NAME = 'deepseek-r1:8b'  # или любая доступная модель

# Параметры запроса
REQUEST_TIMEOUT = (10, 300)  # (connect timeout, read timeout) в секундах
MAX_RETRIES = 5
RETRY_BACKOFF_FACTOR = 2  # множитель для задержки между попытками
STREAM_CHUNK_SIZE = 1024

# Путь к файлу с компетенциями
COMPETENCIES_FILE = Path("MentorComps.txt")


def check_ollama_health():
    """
    Verify that the ollama server is reachable and authentication works.
    Uses the /api/tags endpoint which lists available models.
    Returns True if healthy, False otherwise.
    """
    url = f"{OLLAMA_BASE_URL}/api/tags"
    try:
        response = requests.get(url, auth=AUTH, timeout=10)
        response.raise_for_status()
        data = response.json()
        models = [m["name"] for m in data.get("models", [])]
        print(f"Ollama server OK. Available models: {models}")
        return True
    except Exception as e:
        print(f"Ollama health check failed: {e}")
        return False
    
#Test the oLLAMA check method:
check_ollama_health()

Ollama server OK. Available models: ['gemma3:4b', 'deepseek-r1:8b', 'qwen3.5:9b']


True

In [11]:
def load_competencies(file_path: Path) -> str:
    """Загружает текст компетенций из файла."""
    if not file_path.exists():
        raise FileNotFoundError(f"Файл компетенций не найден: {file_path}")
    with open(file_path, 'r', encoding='utf-8') as f:
        content = f.read().strip()
    logger.info(f"Компетенции загружены, длина текста: {len(content)} символов")
    return content

competencies_text = load_competencies(COMPETENCIES_FILE)

2026-04-21 18:05:25,554 - INFO - Компетенции загружены, длина текста: 493 символов


In [ ]:
def check_ollama_ready(base_url: str, auth: tuple, timeout: int = 10) -> bool:
    """
    Проверяет готовность Ollama-сервера.
    Использует эндпоинт /api/tags, который возвращает список моделей.
    """
    try:
        response = requests.get(
            f"{base_url}/api/tags",
            auth=auth,
            timeout=timeout,
            verify=True  # для HTTPS обязательно
        )
        if response.status_code == 200:
            logger.info("Ollama-сервер доступен")
            return True
        else:
            logger.warning(f"Сервер ответил статусом {response.status_code}: {response.text}")
            return False
    except requests.exceptions.RequestException as e:
        logger.warning(f"Ошибка подключения к серверу: {e}")
        return False

def generate_with_retry(
    base_url: str,
    auth: tuple,
    prompt: str,
    model: str,
    timeout: tuple,
    max_retries: int,
    backoff_factor: int
) -> str:
    """
    Отправляет запрос на генерацию в Ollama с повторными попытками и стримингом.
    Возвращает полный сгенерированный текст.
    """
    url = f"{base_url}/api/generate"
    payload = {
        "model": model,
        "prompt": prompt,
        "stream": True,
        "options": {
            "temperature": 0.1,
            "num_predict": -1  # без ограничения длины
        }
    }
    
    for attempt in range(1, max_retries + 1):
        try:
            logger.info(f"Попытка {attempt}/{max_retries}: отправка запроса...")
            
            # Проверяем готовность сервера перед запросом (опционально)
            if not check_ollama_ready(base_url, auth, timeout=timeout[0]):
                raise ConnectionError("Сервер не готов")
            
            response = requests.post(
                url,
                json=payload,
                auth=auth,
                timeout=timeout,
                stream=True,
                verify=True
            )
            response.raise_for_status()
            
            # Обработка стримингового ответа
            full_text = ""
            for line in response.iter_lines(decode_unicode=True, chunk_size=STREAM_CHUNK_SIZE):
                if line:
                    try:
                        data = json.loads(line)
                        chunk = data.get("response", "")
                        full_text += chunk
                        # Опционально: выводим прогресс
                        if data.get("done", False):
                            break
                    except json.JSONDecodeError:
                        logger.warning(f"Не удалось распарсить строку: {line}")
                        continue
            
            logger.info(f"Генерация завершена успешно, длина ответа: {len(full_text)} символов")
            return full_text
            
        except requests.exceptions.Timeout as e:
            logger.error(f"Таймаут запроса: {e}")
        except requests.exceptions.ConnectionError as e:
            logger.error(f"Ошибка соединения: {e}")
        except requests.exceptions.RequestException as e:
            logger.error(f"Ошибка запроса: {e}")
        except Exception as e:
            logger.error(f"Неожиданная ошибка: {e}")
        
        if attempt < max_retries:
            wait_time = backoff_factor ** attempt
            logger.info(f"Ожидание {wait_time} сек перед следующей попыткой...")
            time.sleep(wait_time)
    
    raise RuntimeError(f"Не удалось выполнить генерацию после {max_retries} попыток")

In [13]:
def build_mentor_dialogue_prompt(competencies: str) -> str:
    """Создаёт промпт для генерации диалога ментора и менти."""
    prompt = f"""
Ты — опытный ментор, обладающий следующими компетенциями:
{competencies}

Твоя задача: создать реалистичный диалог между ментором и менти (подопечным), который наглядно демонстрирует применение этих компетенций в работе.

Диалог должен:
- Иллюстрировать не менее трёх компетенций из списка.
- Показывать типичную ситуацию наставничества (например, обсуждение карьерных целей, решение рабочей проблемы, обратная связь).
- Содержать реплики обоих участников, с указанием ролей (Ментор: ..., Менти: ...).
- Быть естественным и полезным для обучения менторов.

Напиши диалог длиной примерно 15-20 реплик. Начни сразу с диалога, без вступительных комментариев.
"""
    return prompt.strip()

# Формируем промпт
prompt = build_mentor_dialogue_prompt(competencies_text)

# Запускаем генерацию
try:
    generated_dialogue = generate_with_retry(
        base_url=OLLAMA_BASE_URL,
        auth=AUTH,
        prompt=prompt,
        model=MODEL_NAME,
        timeout=REQUEST_TIMEOUT,
        max_retries=MAX_RETRIES,
        backoff_factor=RETRY_BACKOFF_FACTOR
    )
    print("=== СГЕНЕРИРОВАННЫЙ ДИАЛОГ ===")
    print(generated_dialogue)
except Exception as e:
    logger.error(f"Генерация не удалась: {e}")
    raise

2026-04-21 18:06:32,027 - INFO - Попытка 1/5: отправка запроса...
2026-04-21 18:06:32,142 - INFO - Ollama-сервер доступен
2026-04-21 18:10:55,696 - INFO - Генерация завершена успешно, длина ответа: 2375 символов


=== СГЕНЕРИРОВАННЫЙ ДИАЛОГ ===
**Менти:** Я всё ещё не уверен, как правильно реагировать на эти изменения в нашей отрасли. Кажется, мне нужно менять свою работу, но я не понимаю, как подойти к этому процессу.  

**Ментор:** Привет, давай разберёмся. Я знаю, что изменения могут быть сложными, но это нормально. Давай начнём с того, что тебе конкретно не ясно? Может, начать с формулировки твоих вопросов или опасений.  

**Менти:** Я думаю, моя текущая роль может исчезнуть через 6 месяцев из-за внедрения новых технологий. Я боюсь, что если я не сделаю что-то сейчас, я упущу возможности.  

**Ментор:** Спасибо, что поделился своими переживаниями. Ты уже сделал первый шаг — осознать, что изменения могут повлиять на твою работу. Как ты думаешь, какие конкретно действия ты можешь предпринять, чтобы не остаться пассивным?  

**Менти:** Я подумываю о переквалификации, но не знаю, с чего начать. Возможно, мне стоит изучить смежные области или найти руководство.  

**Ментор:** Отличная идея. Я пом

In [14]:
# Сохраним результат в файл для дальнейшего использования
output_file = Path("mentor_dialogue_sample1.txt")
output_file.write_text(generated_dialogue, encoding='utf-8')
logger.info(f"Диалог сохранён в {output_file}")

2026-04-21 18:11:47,205 - INFO - Диалог сохранён в mentor_dialogue_sample1.txt


In [15]:
# Пути к файлам
DIALOGUE_FILE = Path("mentor_dialogue_sample1.txt")          # диалог, сгенерированный ранее
COMPETENCE_FILE = Path("Mentor_competence_active_listening.txt")  # расширенное описание компетенц

In [ ]:
'''Второй промт, отправленный в DeepSeek:
Отлично, все работает! А теперь представь, что у меня есть еще расширенные файлы компетенций. 
Например, компетенция Mentor_competence_active_listening.txt. 
Давай попробуем, используя тот же oLLAMA-сервер, напишем Python-скрипт, с помощью которого проанализируем этот диалог на соответствие этим расширенным компетенциям и вывод рекомендаций, что можно ментору улучшить в своем диалоге.'''

In [16]:
def load_text_file(file_path: Path) -> str:
    """Загружает содержимое текстового файла."""
    if not file_path.exists():
        raise FileNotFoundError(f"Файл не найден: {file_path}")
    with open(file_path, 'r', encoding='utf-8') as f:
        content = f.read().strip()
    logger.info(f"Загружен файл {file_path.name}, длина: {len(content)} символов")
    return content

dialogue_text = load_text_file(DIALOGUE_FILE)
competence_description = load_text_file(COMPETENCE_FILE)

2026-04-21 18:23:32,457 - INFO - Загружен файл mentor_dialogue_sample1.txt, длина: 2375 символов
2026-04-21 18:23:32,459 - INFO - Загружен файл Mentor_competence_active_listening.txt, длина: 838 символов


In [17]:
#Функции для работы с Ollama (аналогичны предыдущим, но адаптированы для анализа)
def check_ollama_ready(base_url: str, auth: tuple, timeout: int = 10) -> bool:
    """Проверяет доступность Ollama-сервера."""
    try:
        response = requests.get(
            f"{base_url}/api/tags",
            auth=auth,
            timeout=timeout,
            verify=True
        )
        return response.status_code == 200
    except:
        return False

def generate_with_retry(
    base_url: str,
    auth: tuple,
    prompt: str,
    model: str,
    timeout: tuple,
    max_retries: int,
    backoff_factor: int
) -> str:
    """Отправляет промпт в Ollama с повторными попытками и стримингом ответа."""
    url = f"{base_url}/api/generate"
    payload = {
        "model": model,
        "prompt": prompt,
        "stream": True,
        "options": {
            "temperature": 0.3,      # для анализа лучше пониже
            "num_predict": -1
        }
    }
    
    for attempt in range(1, max_retries + 1):
        try:
            logger.info(f"Попытка {attempt}/{max_retries}: отправка запроса на анализ...")
            if not check_ollama_ready(base_url, auth, timeout=timeout[0]):
                raise ConnectionError("Сервер не готов")
            
            response = requests.post(
                url,
                json=payload,
                auth=auth,
                timeout=timeout,
                stream=True,
                verify=True
            )
            response.raise_for_status()
            
            full_text = ""
            for line in response.iter_lines(decode_unicode=True, chunk_size=STREAM_CHUNK_SIZE):
                if line:
                    try:
                        data = json.loads(line)
                        chunk = data.get("response", "")
                        full_text += chunk
                        if data.get("done", False):
                            break
                    except json.JSONDecodeError:
                        continue
            
            logger.info(f"Анализ завершён, длина ответа: {len(full_text)} символов")
            return full_text
            
        except Exception as e:
            logger.error(f"Ошибка: {e}")
            if attempt < max_retries:
                wait_time = backoff_factor ** attempt
                logger.info(f"Ожидание {wait_time} сек...")
                time.sleep(wait_time)
    
    raise RuntimeError(f"Не удалось выполнить анализ после {max_retries} попыток")

In [18]:
def build_analysis_prompt(dialogue: str, competence_desc: str) -> str:
    """Создаёт промпт для анализа диалога на соответствие компетенции."""
    prompt = f"""
Ты — эксперт по оценке менторских компетенций. Тебе предоставлен диалог между ментором и менти, а также подробное описание одной из ключевых компетенций ментора.

Описание компетенции:
{competence_desc}

Диалог:
{dialogue}

Проанализируй диалог строго с точки зрения проявления указанной компетенции. Выполни следующие шаги:

1. **Оценка соответствия**: Насколько хорошо ментор демонстрирует эту компетенцию в диалоге? Выдели конкретные примеры реплик ментора, которые подтверждают твою оценку.
2. **Упущенные возможности**: Укажи моменты в диалоге, где ментор мог бы применить компетенцию более эффективно, но не сделал этого. Приведи примеры улучшенных реплик.
3. **Рекомендации**: Сформулируй 3-5 конкретных, практических советов для ментора, как усилить проявление данной компетенции в будущих диалогах.

Ответ должен быть структурированным, полезным и написан на русском языке. Начни сразу с анализа, без вступления.
"""
    return prompt.strip()

# Формируем промпт
analysis_prompt = build_analysis_prompt(dialogue_text, competence_description)

# Запускаем генерацию анализа
try:
    analysis_result = generate_with_retry(
        base_url=OLLAMA_BASE_URL,
        auth=AUTH,
        prompt=analysis_prompt,
        model=MODEL_NAME,
        timeout=REQUEST_TIMEOUT,
        max_retries=MAX_RETRIES,
        backoff_factor=RETRY_BACKOFF_FACTOR
    )
    print("=== АНАЛИЗ ДИАЛОГА ===")
    print(analysis_result)
except Exception as e:
    logger.error(f"Анализ не удался: {e}")
    raise

2026-04-21 18:24:15,890 - INFO - Попытка 1/5: отправка запроса на анализ...
2026-04-21 18:35:26,037 - INFO - Анализ завершён, длина ответа: 4887 символов


=== АНАЛИЗ ДИАЛОГА ===
### Анализ диалога с точки зрения компетенции "Активное слушание"

#### 1. **Оценка соответствия**
Ментор демонстрирует активное слушание в диалоге на высоком уровне. Он не только слушает менти, но и активно работает над пониманием его мыслей, чувств и потребностей. Вот конкретные примеры из диалога, подтверждающие это:

- **Перефразирование**:  
  Ментор уточняет и переформулирует высказывания менти, чтобы показать, что он понял суть его беспокойства. Например:  
  - "Ты уже сделал первый шаг — осознать, что изменения могут повлиять на твою работу." (Ментор подтверждает, что менти осознает проблему, что помогает укрепить доверие).  
  - "Ты молодец, что понимаешь, что лучше двигаться поэтапно." (Ментор подтверждает мысль менти о разбивке задачи на этапы, что демонстрирует понимание его подхода).

- **Открытые вопросы**:  
  Ментор задает уточняющие вопросы, чтобы глубже раскрыть мысли и чувства менти. Например:  
  - "Как ты думаешь, какие конкретно действия ты 

In [19]:
report_file = Path("mentor_analysis_report.txt")
report_file.write_text(analysis_result, encoding='utf-8')
logger.info(f"Отчёт сохранён в {report_file}")

2026-04-21 18:36:42,271 - INFO - Отчёт сохранён в mentor_analysis_report.txt


In [ ]:
def analyze_multiple_competencies(dialogue: str, competence_files: list[Path]) -> dict:
    """
    Последовательно анализирует диалог по списку файлов компетенций.
    Возвращает словарь {имя_файла: результат_анализа}.
    """
    results = {}
    for comp_file in competence_files:
        logger.info(f"Анализ по компетенции: {comp_file.name}")
        comp_desc = load_text_file(comp_file)
        prompt = build_analysis_prompt(dialogue, comp_desc)
        result = generate_with_retry(
            base_url=OLLAMA_BASE_URL,
            auth=AUTH,
            prompt=prompt,
            model=MODEL_NAME,
            timeout=REQUEST_TIMEOUT,
            max_retries=MAX_RETRIES,
            backoff_factor=RETRY_BACKOFF_FACTOR
        )
        results[comp_file.stem] = result
        # Пауза между запросами, чтобы не перегружать сервер
        time.sleep(2)
    return results

# Пример использования (раскомментировать при необходимости)
# comp_folder = Path("competencies")
# comp_files = list(comp_folder.glob("Mentor_competence_*.txt"))
# multi_results = analyze_multiple_competencies(dialogue_text, comp_files)
# for name, analysis in multi_results.items():
#     print(f"\n===== {name} =====\n{analysis}")

In [ ]:
'''Третий промт, отправленный в DeepSeek:
Я хочу сгенерировать пример диалога установочной сессии. 
Компетенции ментора, необходимые для установки долгосрочного контракта, находятся в файле Effetcive_pairing.txt. 
Дай мне необходимый код для запуска в Jupyter.'''

In [21]:
# === ПУТИ К ФАЙЛАМ ===
COMPETENCIES_FILE = Path("Effective_pairing.txt")
OUTPUT_DIALOGUE_FILE = Path("pairing_session_dialogue.txt")

def load_text_file(file_path: Path) -> str:
    """Читает текстовый файл и возвращает его содержимое."""
    if not file_path.exists():
        raise FileNotFoundError(f"Файл не найден: {file_path}")
    with open(file_path, 'r', encoding='utf-8') as f:
        content = f.read().strip()
    logger.info(f"Файл '{file_path.name}' загружен, длина: {len(content)} символов")
    return content

competencies_text = load_text_file(COMPETENCIES_FILE)

2026-04-23 11:43:39,807 - INFO - Файл 'Effective_pairing.txt' загружен, длина: 1059 символов


In [22]:
def build_pairing_session_prompt(competencies: str) -> str:
    """Формирует промпт для создания диалога установочной сессии ментора и менти."""
    prompt = f"""
Ты — опытный ментор, который проводит установочную сессию с новым менти.
Твои компетенции, необходимые для установки долгосрочного контракта, описаны ниже:

{competencies}

На основе этих компетенций сгенерируй реалистичный диалог установочной сессии, в котором:
- Ментор и менти обсуждают цели менторства, ожидания от сотрудничества, формат взаимодействия, правила обратной связи и критерии успеха.
- Явно демонстрируются указанные компетенции (не менее трёх из описания).
- Диалог содержит 15-20 реплик, с указанием ролей (Ментор: ..., Менти: ...).

Диалог должен быть живым, естественным и служить хорошим примером для других менторов. Начни сразу с диалога без вступительных комментариев.
"""
    return prompt.strip()

# Формируем промпт
pairing_prompt = build_pairing_session_prompt(competencies_text)

# Запускаем генерацию
try:
    generated_dialogue = generate_with_retry(
        base_url=OLLAMA_BASE_URL,
        auth=AUTH,
        prompt=pairing_prompt,
        model=MODEL_NAME,
        timeout=REQUEST_TIMEOUT,
        max_retries=MAX_RETRIES,
        backoff_factor=RETRY_BACKOFF_FACTOR
    )
    print("\n=== СГЕНЕРИРОВАННЫЙ ДИАЛОГ УСТАНОВОЧНОЙ СЕССИИ ===\n")
    print(generated_dialogue)
except Exception as e:
    logger.error(f"Не удалось сгенерировать диалог: {e}")
    raise

2026-04-23 11:44:04,433 - INFO - Попытка 1/5: отправка запроса на анализ...
2026-04-23 11:49:13,771 - INFO - Анализ завершён, длина ответа: 2669 символов



=== СГЕНЕРИРОВАННЫЙ ДИАЛОГ УСТАНОВОЧНОЙ СЕССИИ ===

**Ментор:** Привет, [Имя Менти]. Рад встретить тебя сегодня. Я вижу, ты проявляешь интерес к менторству, и я готов поделиться своим опытом. Давай начнем с главного: чем ты ожидаешь от этого процесса?

**Менти:** Привет, [Имя Ментор]. Я надеюсь получить не только практические советы, но и понять, как развивать свои навыки в твоей сфере. Мне важно, чтобы мы вместе определили цели и пути их достижения.

**Ментор:** Отлично. Тогда давай сначала обсудим цели. Какие конкретные задачи ты ставишь перед собой в рамках нашего сотрудничества?

**Менти:** Мне интересно разобраться в управлении проектами и построении команд. Также я хочу узнать, как эффективно взаимодействовать с клиентами в сложных ситуациях.

**Ментор:** Я вижу, что ты уже определил несколько направлений. Это хорошо. Но давай уточним, что ты ожидешь от меня как ментора? Какие ожидания у тебя от этого процесса?

**Менти:** Я хочу, чтобы ты давал конкретные советы и помогал мне а

In [23]:
#Сохранение результата (опционально)
try:
    OUTPUT_DIALOGUE_FILE.write_text(generated_dialogue, encoding='utf-8')
    logger.info(f"Диалог сохранён в файл: {OUTPUT_DIALOGUE_FILE}")
except NameError:
    logger.warning("Переменная generated_dialogue не определена (возможно, генерация не удалась).")

2026-04-23 12:23:48,547 - INFO - Диалог сохранён в файл: pairing_session_dialogue.txt
